In [ ]:
import torch
print("torch:", torch.__version__)
try:
    import torchaudio
    print("torchaudio:", torchaudio.__version__)
except Exception as e:
    print("torchaudio issue:", e)

torch: 2.11.0+cu128
torchaudio: 2.11.0+cu128


In [ ]:
import importlib.util
spec = importlib.util.find_spec("torchaudio")
print("find_spec result:", spec)

import subprocess
result = subprocess.run(["pip", "show", "torchaudio"], capture_output=True, text=True)
print("pip show output:", result.stdout, result.stderr)

find_spec result: ModuleSpec(name='torchaudio', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7a7325c4e930>, origin='/usr/local/lib/python3.13/dist-packages/torchaudio/__init__.py', submodule_search_locations=['/usr/local/lib/python3.13/dist-packages/torchaudio'])
pip show output: Name: torchaudio
Version: 2.11.0+cu128
Summary: An audio package for PyTorch
Home-page: https://github.com/pytorch/audio
Author: Soumith Chintala, David Pollack, Sean Naren, Peter Goldsborough, Moto Hira, Caroline Chen, Jeff Hwang, Zhaoheng Ni, Xiaohui Zhang
Author-email: soumith@pytorch.org
License: 
Location: /usr/local/lib/python3.13/dist-packages
Requires: 
Required-by: 
 


In [ ]:
!pip install torchaudio -q

STEP 2: MODEL LOAD

In [ ]:
import sys
import types
import importlib.machinery

fake_torchaudio = types.ModuleType("torchaudio")
fake_torchaudio.__spec__ = importlib.machinery.ModuleSpec("torchaudio", loader=None)
sys.modules["torchaudio"] = fake_torchaudio

from transformers.models.qwen2_vl.modeling_qwen2_vl import Qwen2VLForConditionalGeneration
from transformers import AutoProcessor
import torch

model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")
print("Model loaded successfully")

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/56.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Model loaded successfully


STEP 3: SCHEMA + PARSER


In [ ]:
from pydantic import BaseModel
import json
import re

class RoomAnalysis(BaseModel):
    room_type: str
    furniture: list[str]
    decor: list[str]
    colors: list[str]
    style: str
    lighting: str
    spatial_observations: list[str]

def parse_room_analysis(raw_text: str) -> RoomAnalysis:
    cleaned = re.sub(r"^```(json)?\s*|\s*```$", "", raw_text.strip())
    data = json.loads(cleaned)
    return RoomAnalysis(**data)

STEP 4 — the analyze_room function

In [ ]:
from PIL import Image
import gc

def analyze_room(image: Image.Image) -> RoomAnalysis:
    structured_prompt = """Analyze this room and respond with ONLY a valid JSON object, no other text, in exactly this format:

{
  "room_type": "string",
  "furniture": ["item1", "item2"],
  "decor": ["item1", "item2"],
  "colors": ["color1", "color2"],
  "style": "string",
  "lighting": "string",
  "spatial_observations": ["observation1", "observation2"]
}

"furniture" is functional items (seating, tables, storage). "decor" is non-furniture items (art, mirrors, plants). "spatial_observations" describes positions and layout, e.g. "bookcase on left wall", "door on right side".

Do not include any explanation, markdown formatting, or text outside the JSON object."""

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": structured_prompt}
            ]
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=250)

    result = processor.batch_decode(output, skip_special_tokens=True)
    raw_output = result[0].split("assistant\n")[-1]

    del inputs, output
    gc.collect()
    torch.cuda.empty_cache()

    return parse_room_analysis(raw_output)

STEP 5 — upload and run:

In [ ]:
from google.colab import files
import io

uploaded = files.upload()
filename = list(uploaded.keys())[0]
image = Image.open(io.BytesIO(uploaded[filename])).convert("RGB")

max_dim = 1024
if max(image.size) > max_dim:
    ratio = max_dim / max(image.size)
    image = image.resize((int(image.size[0]*ratio), int(image.size[1]*ratio)))

result = analyze_room(image)
print(result)

Saving room1.jpg to room1 (2).jpg
room_type='Bedroom' furniture=['bed', 'table', 'nightstand'] decor=['paintings', 'wooden wardrobe'] colors=['beige', 'brown'] style='Traditional' lighting='table lamps' spatial_observations=['bed against the wall', 'nightstand on the left side']
